In [ ]:
# Step 1 — Download JRC Global Surface Water (Permanent Water Mask)

# # // GEE — export JRC permanent water mask per chip
# # // Run this in GEE for each USA chip bbox

# var jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
#             .select("seasonality");

# // Permanent water = present ≥10 months/year
# var permanent_water = jrc.gte(10).rename("permanent_water");

# var aoi = ee.Geometry.Rectangle([-95.53, 38.63, -95.48, 38.68]);

# Export.image.toDrive({
#   image:          permanent_water.clip(aoi),
#   description:    "JRC_permanent_water_USA_955053",
#   folder:         "sen1floods11_masks",
#   fileNamePrefix: "JRC_USA_955053",
#   region:         aoi,
#   scale:          10,
#   crs:            "EPSG:4326",
#   dimensions:     "512x512",
#   maxPixels:      1e13
# });

In [ ]:
# Step 1 — Download JRC Global Surface Water (Permanent Water Mask)

import planetary_computer
import pystac_client
import odc.stac
import rioxarray
from pyproj import Transformer
from pathlib import Path

def download_jrc_for_chip(chip_id, min_lon, min_lat, max_lon, max_lat, out_dir="jrc_masks/"):
    """
    Downloads JRC Global Surface Water seasonality band
    clipped to chip bbox from Microsoft Planetary Computer.
    """
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace
    )

    # Search for JRC GSW
    search = catalog.search(
        collections=["jrc-gsw"],
        bbox=[min_lon, min_lat, max_lon, max_lat]
    )
    items = list(search.items())
    if not items:
        print(f"No JRC data found for {chip_id}")
        return None

    print(f"Found {len(items)} JRC items for {chip_id}")

    # Load seasonality band only
    ds = odc.stac.load(
        items,
        bands=["seasonality"],
        bbox=[min_lon, min_lat, max_lon, max_lat],
        crs="EPSG:4326",
        resolution=0.0000898   # ~10m in degrees
    )

    # Permanent water = seasonality >= 10
    permanent = (ds["seasonality"].values[0] >= 10).astype("uint8")

    # Save as GeoTIFF
    import rasterio
    from rasterio.transform import from_bounds
    Path(out_dir).mkdir(exist_ok=True)
    out_path = Path(out_dir) / f"JRC_{chip_id}.tif"

    transform = from_bounds(min_lon, min_lat, max_lon, max_lat,
                            permanent.shape[1], permanent.shape[0])
    with rasterio.open(
        out_path, "w",
        driver="GTiff", height=permanent.shape[0], width=permanent.shape[1],
        count=1, dtype="uint8", crs="EPSG:4326", transform=transform
    ) as dst:
        dst.write(permanent, 1)

    print(f"✓ Saved → {out_path}  shape={permanent.shape}")
    return out_path

In [16]:
# ── Run across all selected USA chips ─────────────────────────────────────────
import pandas as pd
df = pd.read_csv("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/generated/usa_chips_with_metadata_15_percent_flood_cropland.csv")
for _, row in df.iterrows():
    download_jrc_for_chip(
        chip_id  = row["chip_id"],
        min_lon  = row["min_lon"],
        min_lat  = row["min_lat"],
        max_lon  = row["max_lon"],
        max_lat  = row["max_lat"],
        # out_dir  = "generated/jrc_masks/"
    )

Found 1 JRC items for USA_430764
✓ Saved → jrc_masks/JRC_USA_430764.tif  shape=(513, 513)
Found 1 JRC items for USA_955053
✓ Saved → jrc_masks/JRC_USA_955053.tif  shape=(513, 513)
Found 1 JRC items for USA_58086
✓ Saved → jrc_masks/JRC_USA_58086.tif  shape=(512, 512)
Found 1 JRC items for USA_86502
✓ Saved → jrc_masks/JRC_USA_86502.tif  shape=(513, 513)
Found 1 JRC items for USA_170264
✓ Saved → jrc_masks/JRC_USA_170264.tif  shape=(513, 514)
Found 1 JRC items for USA_595451
✓ Saved → jrc_masks/JRC_USA_595451.tif  shape=(512, 513)
Found 1 JRC items for USA_217598
✓ Saved → jrc_masks/JRC_USA_217598.tif  shape=(513, 513)
Found 1 JRC items for USA_826217
✓ Saved → jrc_masks/JRC_USA_826217.tif  shape=(512, 513)
Found 2 JRC items for USA_646878
✓ Saved → jrc_masks/JRC_USA_646878.tif  shape=(513, 512)
Found 1 JRC items for USA_1068362
✓ Saved → jrc_masks/JRC_USA_1068362.tif  shape=(513, 513)
Found 1 JRC items for USA_986268
✓ Saved → jrc_masks/JRC_USA_986268.tif  shape=(512, 513)
Found 1 JRC 

In [18]:
# Step 2 — Build the Remapping Function
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from pathlib import Path

CROPLAND_CODES = set(range(1, 62)) | set(range(66, 78)) | set(range(204, 255))

def build_agricultural_damage_label(
    label_path,      # Sen1Floods11 LabelHand .tif
    cdl_path,        # USDA CDL clipped to AOI
    jrc_path,        # JRC permanent water mask
    out_path,        # output remapped label .tif
):
    """
    Remaps Sen1Floods11 labels to agricultural damage label space:
        0 = ignore  (invalid, permanent water, non-cropland)
        1 = non-flooded cropland
        2 = flooded cropland
    """
    # ── Load label ────────────────────────────────────────────────────────────
    with rasterio.open(label_path) as src:
        label     = src.read(1).astype("int16")   # -1, 0, 1
        transform = src.transform
        crs       = src.crs
        profile   = src.profile.copy()
        shape     = label.shape                    # (512, 512)

    # ── Load and reproject CDL to chip grid ───────────────────────────────────
    with rasterio.open(cdl_path) as src:
        cdl_raw = src.read(1)
        cdl_reproj = np.zeros(shape, dtype=np.uint8)
        reproject(
            source=cdl_raw, destination=cdl_reproj,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=transform,     dst_crs=crs,
            resampling=Resampling.nearest
        )

    # ── Load and reproject JRC to chip grid ───────────────────────────────────
    with rasterio.open(jrc_path) as src:
        jrc_raw = src.read(1)
        jrc_reproj = np.zeros(shape, dtype=np.uint8)
        reproject(
            source=jrc_raw, destination=jrc_reproj,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=transform,     dst_crs=crs,
            resampling=Resampling.nearest
        )

    # ── Build masks ───────────────────────────────────────────────────────────
    invalid_mask   = (label == -1)
    flood_mask     = (label == 1)
    noflood_mask   = (label == 0)
    cropland_mask  = np.isin(cdl_reproj, list(CROPLAND_CODES))
    permwater_mask = (jrc_reproj == 1)

    # ── Apply remapping logic ─────────────────────────────────────────────────
    output = np.zeros(shape, dtype=np.int16)   # default: ignore (0)

    # Class 2 — flooded cropland (what you want to detect)
    output[flood_mask & cropland_mask & ~permwater_mask] = 2

    # Class 1 — non-flooded cropland (negative class)
    output[noflood_mask & cropland_mask] = 1

    # Everything else stays 0 (ignore):
    # - invalid pixels
    # - permanent water
    # - flood over non-cropland
    # - non-flood over non-cropland

    # ── Save ──────────────────────────────────────────────────────────────────
    profile.update(dtype="int16", count=1, nodata=0)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(output, 1)

    # ── Stats ─────────────────────────────────────────────────────────────────
    print(f"✓ {Path(label_path).stem}")
    print(f"  ignore (0)            : {np.sum(output==0):,}")
    print(f"  non-flooded crop (1)  : {np.sum(output==1):,}")
    print(f"  flooded crop (2)      : {np.sum(output==2):,}")
    print(f"  flood retention rate  : "
          f"{100*np.sum(output==2)/max(np.sum(flood_mask),1):.1f}% "
          f"of original flood pixels kept")

    return output

In [20]:
# Step 3 — Run Across All Selected USA Chips
import pandas as pd

filtered  = pd.read_csv("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/generated/usa_chips_with_metadata_15_percent_flood_cropland.csv")
label_dir = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/Sen1Floods11_data/v1.1/data/flood_events/HandLabeled/LabelHand")
cdl_path  = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/generated/cdl_usa_flood.tif")
jrc_dir   = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/generated/jrc_masks")
out_dir   = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/generated/labels_remapped/")

for _, row in filtered.iterrows():
    chip_id    = row["chip_id"]
    label_path = label_dir / f"{chip_id}_LabelHand.tif"
    jrc_path   = jrc_dir   / f"JRC_{chip_id}.tif"

    if not jrc_path.exists():
        print(f"WARNING: JRC mask missing for {chip_id} — skipping")
        continue

    build_agricultural_damage_label(
        label_path = label_path,
        cdl_path   = cdl_path,
        jrc_path   = jrc_path,
        out_path   = out_dir / f"{chip_id}_LabelAgDamage.tif"
    )

✓ USA_430764_LabelHand
  ignore (0)            : 76,823
  non-flooded crop (1)  : 171,608
  flooded crop (2)      : 13,713
  flood retention rate  : 88.8% of original flood pixels kept
✓ USA_955053_LabelHand
  ignore (0)            : 145,241
  non-flooded crop (1)  : 89,580
  flooded crop (2)      : 27,323
  flood retention rate  : 76.2% of original flood pixels kept
✓ USA_58086_LabelHand
  ignore (0)            : 184,289
  non-flooded crop (1)  : 65,596
  flooded crop (2)      : 12,259
  flood retention rate  : 77.1% of original flood pixels kept
✓ USA_86502_LabelHand
  ignore (0)            : 111,707
  non-flooded crop (1)  : 100,487
  flooded crop (2)      : 49,950
  flood retention rate  : 75.3% of original flood pixels kept
✓ USA_170264_LabelHand
  ignore (0)            : 204,917
  non-flooded crop (1)  : 25,085
  flooded crop (2)      : 32,142
  flood retention rate  : 73.8% of original flood pixels kept
✓ USA_595451_LabelHand
  ignore (0)            : 192,577
  non-flooded crop 